In [106]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

### We will be using isolation forest to detect the anamolies

In [107]:
data = pd.read_parquet('../updatedtop20datasets/META.parquet')
data.head(5)

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,Symbol,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year,volatility_diff,volume_z_diff
0,2012-05-18 00:00:00+00:00,38.2318,45.00,38.00,42.05,573576400,37.937002,44.653014,37.706990,41.725761,573576400,0.0,1.0,META,0.000000,0.0,0.0,0.183094,0.0,0.0,0,2012,0.0,0.0
1,2012-05-21 00:00:00+00:00,34.0300,36.66,33.00,36.53,168192700,33.767602,36.377322,32.745544,36.248325,168192700,0.0,1.0,META,-0.109903,0.0,0.0,0.107552,0.0,0.0,0,2012,0.0,0.0
2,2012-05-22 00:00:00+00:00,31.0000,33.59,30.94,32.61,101786600,30.760965,33.330994,30.701428,32.358551,101786600,0.0,1.0,META,-0.089039,0.0,0.0,0.085484,0.0,0.0,0,2012,0.0,0.0
3,2012-05-23 00:00:00+00:00,32.0000,32.50,31.36,31.37,73600000,31.753255,32.249399,31.118189,31.128112,73600000,0.0,1.0,META,0.032258,0.0,0.0,0.035625,0.0,0.0,0,2012,0.0,0.0
4,2012-05-24 00:00:00+00:00,33.0300,33.21,31.77,32.95,50237200,32.775312,32.953925,31.525028,32.695929,50237200,0.0,1.0,META,0.032187,0.0,0.0,0.043597,0.0,0.0,0,2012,0.0,0.0


In [108]:
def data_partition(symbol):
    path = '../updatedtop20datasets/'
    filepath = f'{path}{symbol}.parquet'
    data = pd.read_parquet(filepath)
    columns =  ['price_return', 'volatility_diff', 'Volatility_30Days', 'Price_Swing', 'volume_z_diff', 'volume_zscore30Days', 'MASignal']
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data[columns])
    scaled_data = pd.DataFrame(scaled_data, columns = columns, index = data['date'])
    scaled_data = scaled_data.reset_index()
    return scaled_data

In [122]:
def iso_runner(data,features, contamination = 0.05):
    iso = IsolationForest(contamination=contamination, random_state=42)
    iso.fit(data[features])
    scores = iso.decision_function(data[features])
    return scores

price_features = ['price_return', 'Price_Swing']
volume_features = ['volume_z_diff', 'volume_zscore30Days']
volatiltiy_features = ['volatility_diff', 'Volatility_30Days']
data['price_score'] = iso_runner(data, price_features)
data['volume_score'] = iso_runner(data, volume_features)
data['volatility_score'] = iso_runner(data, volatiltiy_features)
data['cum_score'] = (data['price_score'] + data['volume_score'] + data['volatility_score']) / 3
data['is_anomaly'] = (data['cum_score'] < data['cum_score'].quantile(0.05)).astype(int)
print(data['is_anomaly'].value_counts())

is_anomaly
0    3350
1     177
Name: count, dtype: int64


In [123]:
data.head(5)

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,Symbol,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year,volatility_diff,volume_z_diff,is_anomaly,price_score,volume_score,volatility_score,cum_score
0,2012-05-18 00:00:00+00:00,38.2318,45.00,38.00,42.05,573576400,37.937002,44.653014,37.706990,41.725761,573576400,0.0,1.0,META,0.000000,0.0,0.0,0.183094,0.0,0.0,0,2012,0.0,0.0,1,-0.181362,0.181175,0.133157,0.044323
1,2012-05-21 00:00:00+00:00,34.0300,36.66,33.00,36.53,168192700,33.767602,36.377322,32.745544,36.248325,168192700,0.0,1.0,META,-0.109903,0.0,0.0,0.107552,0.0,0.0,0,2012,0.0,0.0,1,-0.201450,0.181175,0.133157,0.037627
2,2012-05-22 00:00:00+00:00,31.0000,33.59,30.94,32.61,101786600,30.760965,33.330994,30.701428,32.358551,101786600,0.0,1.0,META,-0.089039,0.0,0.0,0.085484,0.0,0.0,0,2012,0.0,0.0,1,-0.159777,0.181175,0.133157,0.051518
3,2012-05-23 00:00:00+00:00,32.0000,32.50,31.36,31.37,73600000,31.753255,32.249399,31.118189,31.128112,73600000,0.0,1.0,META,0.032258,0.0,0.0,0.035625,0.0,0.0,0,2012,0.0,0.0,0,0.121289,0.181175,0.133157,0.145207
4,2012-05-24 00:00:00+00:00,33.0300,33.21,31.77,32.95,50237200,32.775312,32.953925,31.525028,32.695929,50237200,0.0,1.0,META,0.032187,0.0,0.0,0.043597,0.0,0.0,0,2012,0.0,0.0,0,0.082337,0.181175,0.133157,0.132223


In [125]:
peter = data[data['is_anomaly'] == 1]
peter.describe()

,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year,volatility_diff,volume_z_diff,is_anomaly,price_score,volume_score,volatility_score,cum_score
count,177.000000,177.000000,177.000000,177.000000,1.770000e+02,177.000000,177.000000,177.000000,177.000000,1.770000e+02,177.000000,177.0,177.000000,177.000000,177.000000,177.000000,177.000000,177.000000,177.000000,177.000000,177.000000,177.000000,177.0,177.000000,177.000000,177.000000,177.000000
mean,302.341699,309.991605,295.499893,302.960491,6.465095e+07,300.958896,308.570645,294.150735,301.573466,6.465095e+07,0.002966,1.0,-0.000045,11.844196,21.149872,0.057227,1.126169,1.855153,0.355932,2020.745763,-9.305675,-0.728984,1.0,-0.031782,0.005978,0.082209,0.018802
std,223.231902,227.290510,219.378915,223.237779,6.454378e+07,222.861286,226.915880,219.011758,222.866163,6.454378e+07,0.039461,0.0,0.076651,10.636627,17.567126,0.028277,1.117086,1.755437,0.480153,4.312745,16.543376,1.297497,0.0,0.105415,0.086524,0.111823,0.038807
min,19.870000,20.480000,19.690000,20.100000,6.743473e+06,19.716787,20.322083,19.538174,19.945013,6.743473e+06,0.000000,1.0,-0.263901,0.000000,0.000000,0.010439,-1.921355,-1.585077,0.000000,2012.000000,-50.133389,-2.851800,1.0,-0.248356,-0.177292,-0.144556,-0.123880
25%,146.010000,148.180000,137.100000,141.210000,2.983076e+07,144.884147,147.037414,136.042850,140.121159,2.983076e+07,0.000000,1.0,-0.046131,5.025277,6.714802,0.039487,0.351718,0.179096,0.000000,2018.000000,-17.637571,-1.594441,1.0,-0.114919,-0.036876,-0.018265,0.003974
50%,207.600000,211.751700,202.220000,209.750000,4.369286e+07,205.999239,210.118926,200.660723,208.132661,4.369286e+07,0.000000,1.0,-0.010886,8.670016,15.833909,0.052530,1.554489,1.786714,0.000000,2022.000000,-2.198176,-1.098366,1.0,-0.043963,0.013624,0.112863,0.028744
75%,531.620000,547.430000,511.600000,540.100000,7.755693e+07,529.985028,545.746405,508.701494,540.100000,7.755693e+07,0.000000,1.0,0.048249,14.167427,35.880098,0.073758,2.039824,3.399619,1.000000,2025.000000,1.304006,0.265343,1.0,0.050216,0.058866,0.177486,0.046387
max,773.440000,784.750000,765.510000,775.200000,5.735764e+08,771.637872,782.921519,763.726349,773.393771,5.735764e+08,0.525000,1.0,0.296077,52.757276,59.349408,0.184432,2.260472,5.086715,1.000000,2026.000000,23.761699,2.289011,1.0,0.191067,0.183116,0.249996,0.059804


In [126]:
parker = data[data['is_anomaly'] == 0]
parker.describe()

,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year,volatility_diff,volume_z_diff,is_anomaly,price_score,volume_score,volatility_score,cum_score
count,3350.000000,3350.000000,3350.000000,3350.000000,3.350000e+03,3350.000000,3350.000000,3350.000000,3350.000000,3.350000e+03,3350.000000,3350.0,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.0,3350.000000,3350.000000,3350.000000,3350.000000
mean,229.251631,231.922122,226.433792,229.188542,2.598935e+07,227.904154,230.558811,225.103108,227.841774,2.598935e+07,0.001224,1.0,0.001171,4.633128,9.557610,0.024881,-0.091473,-0.105867,0.640299,2018.791343,-4.924482,0.014395,0.0,0.146104,0.134560,0.191472,0.157378
std,186.724106,188.839999,184.532436,186.827240,1.901467e+07,186.232512,188.342478,184.047332,186.336011,1.901467e+07,0.025026,0.0,0.018697,5.260541,9.826540,0.012110,0.947507,0.899489,0.479984,4.029643,7.148213,0.690174,0.0,0.051531,0.057392,0.066570,0.034578
min,17.729000,18.270000,17.550000,18.080000,4.726056e+06,17.592295,18.129124,17.414676,17.940589,4.726056e+06,0.000000,1.0,-0.090551,0.000000,0.000000,0.003715,-2.154895,-2.416273,0.000000,2012.000000,-50.800934,-2.555550,0.0,-0.171380,-0.153764,-0.146133,0.059944
25%,97.002500,97.927500,95.352500,97.005000,1.436644e+07,96.254534,97.172401,94.617256,96.257014,1.436644e+07,0.000000,1.0,-0.009489,1.323800,2.690411,0.016232,-0.779457,-0.672193,0.000000,2015.000000,-6.412265,-0.453916,0.0,0.129302,0.107188,0.181001,0.134919
50%,174.625000,176.655000,172.315000,174.500000,2.010764e+07,173.278503,175.292850,170.986314,173.154466,2.010764e+07,0.000000,1.0,0.001081,2.884750,6.185667,0.022503,-0.313387,-0.310088,1.000000,2019.000000,-2.584933,-0.023809,0.0,0.163434,0.151417,0.214757,0.165590
75%,300.195000,304.900000,296.840000,300.386250,3.006303e+07,297.880258,302.548979,294.551128,298.070033,3.006303e+07,0.000000,1.0,0.012362,5.987571,12.060243,0.031018,0.499255,0.217421,1.000000,2022.000000,-0.833626,0.419174,0.0,0.183059,0.179813,0.233747,0.184660
max,790.000000,796.250000,780.820000,791.150000,2.398240e+08,788.159287,794.394724,779.000676,789.306607,2.398240e+08,0.525000,1.0,0.080923,57.965498,59.288902,0.146136,2.256750,4.729619,1.000000,2026.000000,24.856223,2.532857,0.0,0.200089,0.204973,0.253422,0.212349
